# Data Prep
Prepare data for analysis.

In [13]:
# imports
import pandas as pd

In [14]:
# load datasets in
electricity = pd.read_csv("../data/combined_electricity_data.csv")
print(f"Electricity {electricity.columns}")

water = pd.read_csv("../data/combined_water_data.csv")
print(f"Water {water.columns}")

construction = pd.read_csv("../data/construction_data.csv")
print(f"Construction {construction.columns}")

Electricity Index(['state', 'year', 'sector', 'electricity_usage'], dtype='object')
Water Index(['State', 'Year', 'Water_Usage'], dtype='object')
Construction Index(['state_code', 'asof_month', 'asof_month_yyyy_mm',
       'q_under_construction_facilities', 's_under_construction_power_mw',
       's_under_construction_square_footage', 'q_announcements',
       's_announcements_power_capacity', 's_announcements_square_footage',
       'q_activations', 's_activations_power_capacity',
       's_activations_square_footage', 'updated_at', 'Unnamed: 13'],
      dtype='object')


### Merge datasets on state, year


Standardize column names. 
- Electricity looks good 
- Water just needs lowercase
- Construction needs some remapping. Year needs extracted, state code needs extracted

In [15]:
# Standardize column names
water.columns = water.columns.str.lower()

In [16]:
# construction year needs extracted 
print(construction['asof_month_yyyy_mm'][0])

# dd/mm/yy --> yyyy
construction['year'] = pd.to_datetime(
    construction['asof_month_yyyy_mm'],
    format='%m/%d/%y'
    ).dt.year

1/1/26


In [17]:
# construction state code needs parsed to state
us_state_map = state_map = {
    "AL": "Alabama",
    "AK": "Alaska",
    "AZ": "Arizona",
    "AR": "Arkansas",
    "CA": "California",
    "CO": "Colorado",
    "CT": "Connecticut",
    "DE": "Delaware",
    "FL": "Florida",
    "GA": "Georgia",
    "HI": "Hawaii",
    "ID": "Idaho",
    "IL": "Illinois",
    "IN": "Indiana",
    "IA": "Iowa",
    "KS": "Kansas",
    "KY": "Kentucky",
    "LA": "Louisiana",
    "ME": "Maine",
    "MD": "Maryland",
    "MA": "Massachusetts",
    "MI": "Michigan",
    "MN": "Minnesota",
    "MS": "Mississippi",
    "MO": "Missouri",
    "MT": "Montana",
    "NE": "Nebraska",
    "NV": "Nevada",
    "NH": "New Hampshire",
    "NJ": "New Jersey",
    "NM": "New Mexico",
    "NY": "New York",
    "NC": "North Carolina",
    "ND": "North Dakota",
    "OH": "Ohio",
    "OK": "Oklahoma",
    "OR": "Oregon",
    "PA": "Pennsylvania",
    "RI": "Rhode Island",
    "SC": "South Carolina",
    "SD": "South Dakota",
    "TN": "Tennessee",
    "TX": "Texas",
    "UT": "Utah",
    "VT": "Vermont",
    "VA": "Virginia",
    "WA": "Washington",
    "WV": "West Virginia",
    "WI": "Wisconsin",
    "WY": "Wyoming"
}

construction['state'] = construction['state_code'].map(us_state_map)
print(construction['state'][0])

Georgia


In [ ]:
# clean datacenters years
construction["Year Opened"].dtype
construction['year'] = pd.to_numeric(construction['Year Opened'].str.extract(r'(\d{4})')[0], errors='coerce')
construction["year"].dtype

In [18]:
# clean construction before merge
construction = construction.drop(columns=[
       'state_code',
       'asof_month', 
       'asof_month_yyyy_mm',
       's_under_construction_square_footage', 
       'q_announcements',
       's_announcements_square_footage',
       's_activations_square_footage', 'updated_at', 'Unnamed: 13'
])

In [19]:
# merge datasets
combined = pd.merge(
    electricity,
    water,
    on=['state', 'year'],
    how='outer'
)

combined = pd.merge(
    combined, 
    construction,
    on=['state', 'year'],
    how='outer'
)

combined.head

<bound method NDFrame.head of      state  year       sector  electricity_usage  water_usage  \
0       AK  1990   Commercial          1972116.0          NaN   
1       AK  1990   Industrial           459282.0          NaN   
2       AK  1990  Residential          1661311.0          NaN   
3       AK  1991   Commercial          2005247.0          NaN   
4       AK  1991   Industrial           465878.0          NaN   
...    ...   ...          ...                ...          ...   
9408   NaN  2025          NaN                NaN          NaN   
9409   NaN  2025          NaN                NaN          NaN   
9410   NaN  2025          NaN                NaN          NaN   
9411   NaN  2025          NaN                NaN          NaN   
9412   NaN  2026          NaN                NaN          NaN   

      q_under_construction_facilities  s_under_construction_power_mw  \
0                                 NaN                            NaN   
1                                 NaN        

In [20]:
# filter data for available time range (2000-2020)
combined_filtered = combined[(combined['year'] >= 2000) & (combined['year'] <= 2020)]

In [21]:
import pickle
# save df for access in further notebooks
combined_filtered.to_pickle("../data/merged_state_data_2000_2020.pkl")